# Kaggle Box — run the NVDA pipeline

**Enable Internet, choose an accelerator if available, then Run All.** This notebook clones the public repo and calls its modular `kaggle_v2/train_box.py` entry point. The model, features, news logic, evaluation and recovery remain in separate Python files.

With the defaults, it starts training without API keys or uploaded datasets using the bundled historical market data. `NEWS_MODE="auto"` enables Qwen sentiment when `NEWS_JSONL` or both Alpaca credentials are available. Otherwise it clearly reports a **math-only ablation**, leaves missing-news flags in the features, and trains PPO without downloading Qwen. Use `NEWS_MODE="required"` when a news-enabled run is mandatory.

Qwen remains a frozen news analyzer. PPO learns the trading policy. No orders are placed.


In [ ]:
# Run All with these defaults. Optional settings can be changed before the first run.
REPO_REF = "main"              # first launch resolves and records an exact commit
REFRESH_REPO = False            # use a new CYCLE_ID when adopting code/data changes
CYCLE_ID = "cycle_001"
DEVICE = "auto"                # auto, cpu, cuda (NVIDIA), rocm (AMD), xpu (Intel)
MAX_GPUS = 2                   # dual T4 automatically uses both GPUs
CPU_THREADS = 2
NEWS_MODE = "auto"             # auto, required, off
NEWS_JSONL = ""                # optional attached timestamped news JSONL
DATASET_DIR = ""               # optional replacement bars.json + dataset.json directory
UPDATES = 0                    # 0 = auto: CPU 16; GPU 40
CANDIDATES = 0                 # 0 = auto: CPU 1; GPU 3
MAX_NEWS_ARTICLES = 6000
RUN_FINAL_TEST = False          # train/validate first; reserve the holdout for a deliberate test
RESTORE_FROM = ""              # entire saved nvda-box directory, when starting a new session


In [ ]:
import os, sys, subprocess, signal, shutil, json
from pathlib import Path

REPO_URL = "https://github.com/Cheese-chess-bot/nvda-3b-alpaca-lab.git"
BASE = Path("/kaggle/working/nvda-box") if Path("/kaggle/working").exists() else Path.cwd()/"nvda-box-output"
PROJECT = BASE/"repo"


def execute(command, cwd=None):
    process = subprocess.Popen(list(map(str,command)), cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=os.name!="nt")
    try:
        for line in process.stdout:
            print(line,end="",flush=True)
        status = process.wait()
        if status:
            raise RuntimeError(f"Step failed (exit {status}); read the error above.")
    except BaseException:
        if process.poll() is None:
            if os.name=="nt":
                subprocess.run(["taskkill","/PID",str(process.pid),"/T","/F"],capture_output=True)
            else:
                os.killpg(process.pid,signal.SIGTERM)
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid,signal.SIGKILL)
                    process.wait()
        raise


if RESTORE_FROM:
    if BASE.exists():
        raise RuntimeError("Restore target already exists; do not overwrite an active run.")
    restore=Path(RESTORE_FROM)
    if not (restore/"cycles").is_dir():
        raise RuntimeError("RESTORE_FROM must be the entire saved nvda-box directory.")
    shutil.copytree(restore,BASE)
BASE.mkdir(parents=True,exist_ok=True)
if not shutil.which("git"):
    raise RuntimeError("Git is required. Kaggle includes Git; on a local computer install it first.")
if not PROJECT.exists():
    execute(["git","clone",REPO_URL,PROJECT])
    execute(["git","checkout","--detach",REPO_REF],cwd=PROJECT)
else:
    origin=subprocess.check_output(["git","remote","get-url","origin"],cwd=PROJECT,text=True).strip()
    if origin.rstrip("/")!=REPO_URL:
        raise RuntimeError("This output directory contains a different repository.")
    changed=subprocess.check_output(["git","status","--porcelain"],cwd=PROJECT,text=True).strip()
    if changed:
        raise RuntimeError("The saved checkout has local edits. Preserve them before refreshing or training.")
    if REFRESH_REPO:
        execute(["git","fetch","origin",REPO_REF],cwd=PROJECT)
        execute(["git","checkout","--detach","FETCH_HEAD"],cwd=PROJECT)
revision=subprocess.check_output(["git","rev-parse","HEAD"],cwd=PROJECT,text=True).strip()
print("Repository revision:",revision)
print("This exact revision is retained for resume. Set REFRESH_REPO=True only for a new cycle.")
runner=PROJECT/"kaggle_v2"/"train_box.py"
if not runner.is_file():
    raise RuntimeError("This revision predates Kaggle Box. Select the current main branch in a fresh checkout.")
command=[sys.executable,runner,"--project",PROJECT,"--work-dir",BASE,
    "--cycle",CYCLE_ID,"--device",DEVICE,"--max-gpus",MAX_GPUS,"--threads",CPU_THREADS,
    "--news",NEWS_MODE,"--updates",UPDATES,"--candidates",CANDIDATES,
    "--max-news-articles",MAX_NEWS_ARTICLES]
if NEWS_JSONL:
    command += ["--news-jsonl",NEWS_JSONL]
if DATASET_DIR:
    command += ["--dataset-dir",DATASET_DIR]
if RUN_FINAL_TEST:
    command += ["--final-test"]
execute(command,cwd=PROJECT)
result_path=BASE/"cycles"/CYCLE_ID/"box_result.json"
result=json.loads(result_path.read_text())
print("\nTraining complete:",result["work_dir"])
print("News coverage:",result["news_coverage"])
print("Selected policy validation:",result["winner"]["validation"])
print("Save Version WITH OUTPUTS to keep checkpoints and the holdout ledger.")
from IPython.display import FileLink,display
archive=BASE/"cycles"/CYCLE_ID/"kaggle-box-policy.zip"
display(FileLink(str(archive.relative_to("/kaggle/working")) if Path("/kaggle/working").exists() else str(archive)))


## Device behavior

| Hardware | Training path | Requirements |
|---|---|---|
| NVIDIA GPU | CUDA; NCCL for multiple GPUs | Working CUDA PyTorch build and NVIDIA driver |
| AMD GPU | ROCm using PyTorch's `cuda` device API; NCCL/RCCL for multiple GPUs | A GPU/OS supported by ROCm and a compatible PyTorch build |
| Intel GPU | Native `xpu`; XCCL for multiple GPUs where available | Supported Intel GPU, driver and an XPU-enabled PyTorch build |
| Intel/AMD CPU | Float32 CPU PPO | Works without an accelerator |

The launcher tests actual allocation, forward and backward kernels before training. In `auto` mode an unusable accelerator falls back to CPU with a visible message. An explicitly requested unavailable device stops instead. If multiple GPUs are visible but their distributed backend is missing, it uses one GPU. Kaggle supplies the hardware selected in notebook settings; this file cannot create Intel/AMD GPU instances inside Kaggle.

Dependencies use a separate environment that retains the installed PyTorch build. If PyTorch is missing, a CPU build is installed. An incompatible existing PyTorch version is reported rather than silently replacing a working GPU environment. Install the official accelerator-specific PyTorch build before using a local Intel/AMD/NVIDIA GPU runtime.

The PPO network trains in float32 on every backend. Optional Qwen news analysis uses NVIDIA NF4/float16; AMD/Intel use unquantized float16 and CPU uses float32. Those paths require enough device/host memory for the 3B model. GPU vendor routing is implemented; NVIDIA/ROCm/XPU execution still needs hardware validation.

## News and outputs

For Alpaca news, enable Kaggle Secrets named `APCA_API_KEY_ID` and `APCA_API_SECRET_KEY`; standalone Jupyter can use the corresponding environment variables. The notebook never writes these credentials to output files. A supplied news source must pass timestamp and coverage checks; a failed news fetch is not silently converted to a math-only run. The starter repo has prices, not a supplied historical news corpus.

`nvda-box/cycles/<CYCLE_ID>/` contains the policy weights, per-update checkpoints, validation history, exact source/package/configuration metadata, coverage report and `kaggle-box-policy.zip`. The global champion/holdout ledger is shared across cycles. Keep the **whole output directory** to resume; the policy ZIP alone is not enough. After an interrupted controller, first verify no process is still running before removing a leftover `controller.lock`.

Defaults run training and validation only. `RUN_FINAL_TEST=True` enables the one-use test ledger and research-promotion checks. Already inspected bundled dates cannot promote a champion; genuinely fresh test dates after 2026-10-02 are required. CPU defaults are a smaller starting budget, not a performance claim. No trained policy, future profit or GPU test result is implied by a successful setup.

For future features, edit the separate modules and keep the notebook as the entry point. First launches fetch the chosen `REPO_REF`; later runs retain the exact checkout. To adopt a repo update, set `REFRESH_REPO=True`, choose a new `CYCLE_ID`, and retain the global evaluation ledger.

Sources: [PyTorch ROCm semantics](https://docs.pytorch.org/docs/stable/notes/hip.html), [Intel XPU setup](https://docs.pytorch.org/docs/stable/notes/get_start_xpu.html), [distributed backends](https://docs.pytorch.org/docs/stable/distributed.html), [official PyTorch install selector](https://pytorch.org/get-started/locally/).
